In [14]:
# Jak zmieniła się jakość życia w Polsce - pobieranie danych z Eurostatu do Tableau
# Wymagania: pip install eurostat pandas openpyxl
import eurostat
import pandas as pd

In [15]:
EU27 = ["AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "EL", "ES", "FI", "FR", "HR",
        "HU", "IE", "IT", "LT", "LU", "LV", "MT", "NL", "PL", "PT", "RO", "SE", "SI", "SK",
        "EU27_2020"]

NAZWY = {
    "AT": "Austria",    "BE": "Belgia",      "BG": "Bułgaria",   "CY": "Cypr",       "CZ": "Czechy",
    "DE": "Niemcy",     "DK": "Dania",       "EE": "Estonia",    "EL": "Grecja",     "ES": "Hiszpania",
    "FI": "Finlandia",  "FR": "Francja",     "HR": "Chorwacja",  "HU": "Węgry",      "IE": "Irlandia",
    "IT": "Włochy",     "LT": "Litwa",       "LU": "Luksemburg", "LV": "Łotwa",      "MT": "Malta",
    "NL": "Niderlandy", "PL": "Polska",      "PT": "Portugalia", "RO": "Rumunia",    "SE": "Szwecja",
    "SI": "Słowenia",   "SK": "Słowacja",    "EU27_2020": "UE27 (średnia)",
}

# nazwa kolumny: (zbiór Eurostatu, filtry, startPeriod)
DATASETS = {
    "pkb_pc_pps":     ("nama_10_pc",    {"unit": "CP_PPS_EU27_2020_HAB", "na_item": "B1GQ"}, 2000),
    "placa_min_pps":  ("earn_mw_cur",   {"currency": "PPS"}, "2000-S1"),
    "dlugosc_zycia":  ("demo_mlexpec",  {"unit": "YR", "sex": "T", "age": "Y_LT1"}, 2000),
    "bezrobocie":     ("une_rt_a",      {"unit": "PC_ACT", "age": "Y15-74", "sex": "T"}, 2000),
    "zabojstwa_100k": ("crim_hom_soff", {"unit": "P_HTHAB", "iccs": "ICCS0101", "leg_stat": "PER_VICT", "sex": "T"}, 2000),
    "arop_60":        ("ilc_li02",      {"statinfo": "MED_EI", "unit": "PC", "rskpovth": "B_60", "sex": "T", "age": "TOTAL"},2005),
    "pop_0_14":       ("demo_pjanbroad",{"unit": "NR", "sex": "T", "age": "Y_LT15"},1995),
    "pop_15_64":      ("demo_pjanbroad",{"unit": "NR", "sex": "T", "age": "Y15-64"},1995),
    "pop_65_plus":    ("demo_pjanbroad",{"unit": "NR", "sex": "T", "age": "Y_GE65"},1995)
}


def pobierz(kod, filtry, start):
    dozwolone = set(eurostat.get_par_values(kod, "geo"))
    geo = [g for g in EU27 if g in dozwolone]
    brak = sorted(set(EU27) - dozwolone)
    if brak:
        print(f"{kod}: brak w zbiorze kodów {brak}")

    df = eurostat.get_data_df(kod, filter_pars={"geo": geo, "startPeriod": start, **filtry})
    geo_col = next(c for c in df.columns if str(c).startswith("geo"))
    kolumny_lat = [c for c in df.columns if str(c)[:4].isdigit()]

    d = (df.rename(columns={geo_col: "geo"})
           .melt(id_vars="geo", value_vars=kolumny_lat, var_name="okres", value_name="wartosc")
           .dropna(subset=["wartosc"]))
    d["okres"] = d["okres"].astype(str)
    d["rok"] = d["okres"].str[:4].astype(int)

    # po zawężeniu filtrami każdy kraj ma mieć jedną wartość na okres
    assert not d.duplicated(["geo", "okres"]).any(), f"{kod}: zawęź filtry"

    # dane półroczne (płaca minimalna) -> średnia z roku; reszta bez zmian
    return d.groupby(["geo", "rok"], as_index=False)["wartosc"].mean()


# --- pobranie i połączenie wszystkich wskaźników ---
tabela = None
for nazwa, (kod, filtry, start) in DATASETS.items():
    d = pobierz(kod, filtry, start).rename(columns={"wartosc": nazwa})
    tabela = d if tabela is None else tabela.merge(d, on=["geo", "rok"], how="outer")

tabela.insert(1, "kraj", tabela["geo"].map(NAZWY))
tabela["czy_agregat"] = tabela["geo"] == "EU27_2020"      # do filtrowania w Tableau
tabela = tabela.sort_values(["geo", "rok"]).reset_index(drop=True)

print(tabela.info())

# zakres danych dla Polski (pierwszy i ostatni rok z wartością)
pl = tabela[tabela["geo"] == "PL"]
for w in DATASETS:
    ok = pl.loc[pl[w].notna(), "rok"]
    print(f"{w}: {ok.min()}-{ok.max()}" if len(ok) else f"{w}: brak danych dla PL")

# --- eksport ---
tabela.to_excel("dane_ue.xlsx", index=False)



earn_mw_cur: brak w zbiorze kodów ['EU27_2020']
crim_hom_soff: brak w zbiorze kodów ['EU27_2020']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 880 entries, 0 to 879
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   geo             880 non-null    object 
 1   kraj            880 non-null    object 
 2   rok             880 non-null    int64  
 3   pkb_pc_pps      728 non-null    float64
 4   placa_min_pps   548 non-null    float64
 5   dlugosc_zycia   695 non-null    float64
 6   bezrobocie      486 non-null    float64
 7   zabojstwa_100k  402 non-null    float64
 8   arop_60         572 non-null    float64
 9   pop_0_14        856 non-null    float64
 10  pop_15_64       856 non-null    float64
 11  pop_65_plus     856 non-null    float64
 12  czy_agregat     880 non-null    bool   
dtypes: bool(1), float64(9), int64(1), object(2)
memory usage: 83.5+ KB
None
pkb_pc_pps: 2000-2025
placa_min_pps: 2000-202